# 05 - Extract CNN Features for LSTM Future Risk Model

This notebook uses the trained ResNet50 future-risk model as a feature extractor.

Instead of predicting the final 1 to 5 year risk directly from each image, this notebook extracts CNN feature vectors from each mammogram image. These features will later be grouped by patient and exam session, then passed into an LSTM model for sequential future-risk prediction.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

In [20]:
# Set paths for Colab

CSV_PATH = "/content/drive/MyDrive/EMBED/embed_future_risk_dataset_512.csv"
MODEL_PATH = "/content/drive/MyDrive/EMBED/models/best_resnet50_embed_5yr_risk.pth"

FEATURE_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/Features/embed_cnn_features.npy"
METADATA_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/embed_cnn_feature_metadata.csv"

In [21]:
# Set device

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [22]:
# Load the trained ResNet50 future-risk model

model = models.resnet50(weights=None)

num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 5)
)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

model = model.to(device)
model.eval()

print("Model loaded successfully.")
print("Feature size before final layer:", num_features)

Model loaded successfully.
Feature size before final layer: 2048


In [23]:
feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [24]:
dummy = torch.randn(1, 3, 512, 512).to(device)

with torch.no_grad():
    features = feature_extractor(dummy)

print(features.shape)

torch.Size([1, 2048, 1, 1])


In [25]:
features = features.view(features.size(0), -1)

print(features.shape)

torch.Size([1, 2048])


In [26]:
# Load the final EMBED image dataset

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())
df.head()

Dataset shape: (1038, 13)
['empi_anon', 'acc_anon', 'study_date_anon', 'ViewPosition', 'ImageLateralityFinal', 'processed_image_path', 'future_risk_label', 'days_to_cancer', 'risk_1yr', 'risk_2yr', 'risk_3yr', 'risk_4yr', 'risk_5yr']


,empi_anon,acc_anon,study_date_anon,ViewPosition,ImageLateralityFinal,processed_image_path,future_risk_label,days_to_cancer,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
0,57769289,3512912135438605,2016-06-15 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
1,57769289,3504436559522696,2017-01-21 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
2,57769289,3504436559522696,2017-01-21 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
3,57769289,3512912135438605,2016-06-15 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
4,57769289,6528636244388176,2016-07-20 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1


In [27]:
# Image transform for ResNet50 feature extraction

feature_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [28]:
# Dataset class for extracting CNN features

class EmbedFeatureDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row["processed_image_path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, idx

In [29]:
# Create dataloader

feature_dataset = EmbedFeatureDataset(df, transform=feature_transform)

feature_loader = DataLoader(
    feature_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Total images:", len(feature_dataset))

Total images: 1038


In [30]:
# Extract CNN features for all images

all_features = []

with torch.no_grad():

    for images, indices in feature_loader:

        images = images.to(device)

        features = feature_extractor(images)

        # (batch, 2048, 1, 1)
        features = features.view(features.size(0), -1)

        all_features.append(
            features.cpu().numpy()
        )

cnn_features = np.vstack(all_features)

print("Feature Matrix Shape:")
print(cnn_features.shape)

Feature Matrix Shape:
(1038, 2048)


In [31]:
# Save CNN features

np.save(
    FEATURE_OUTPUT_PATH,
    cnn_features
)

print("Features saved.")
print(FEATURE_OUTPUT_PATH)

Features saved.
/content/drive/MyDrive/EMBED/Features/embed_cnn_features.npy


In [32]:
# Save metadata for sequence creation

metadata_columns = [
    "empi_anon",
    "acc_anon",
    "study_date_anon",
    "ViewPosition",
    "ImageLateralityFinal",
    "processed_image_path",
    "risk_1yr",
    "risk_2yr",
    "risk_3yr",
    "risk_4yr",
    "risk_5yr"
]

feature_metadata = df[
    metadata_columns
].copy()

feature_metadata.to_csv(
    METADATA_OUTPUT_PATH,
    index=False
)

print("Metadata saved.")
print(METADATA_OUTPUT_PATH)

Metadata saved.
/content/drive/MyDrive/EMBED/embed_cnn_feature_metadata.csv


In [33]:
metadata = pd.read_csv(METADATA_OUTPUT_PATH)

patient_exam_counts = (
    metadata.groupby("empi_anon")["study_date_anon"]
    .nunique()
    .reset_index()
)

patient_exam_counts.columns = [
    "patient_id",
    "num_exams"
]

print(patient_exam_counts["num_exams"].describe())

print("\nExam count distribution:")
print(
    patient_exam_counts["num_exams"]
    .value_counts()
    .sort_index()
)

count    40.000000
mean      5.725000
std       3.250148
min       2.000000
25%       3.000000
50%       5.000000
75%       7.000000
max      13.000000
Name: num_exams, dtype: float64

Exam count distribution:
num_exams
2     6
3     6
4     6
5     5
6     3
7     5
8     1
9     2
11    3
12    1
13    2
Name: count, dtype: int64
